In [ ]:
!pip install pdfkit
! pip install weasyprint

In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification


def resize_position_embeddings(model, new_max_pos=1024):
    current_max_pos, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > current_max_pos:
        new_pos_embed = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos_embed.weight.data[:current_max_pos, :] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos_embed.weight.data[current_max_pos:, :] = model.roberta.embeddings.position_embeddings.weight.data[-1, :].repeat(new_max_pos - current_max_pos, 1)
        model.roberta.embeddings.position_embeddings = new_pos_embed
        model.config.max_position_embeddings = new_max_pos
    return model


# Load tokenizer and model with output_attentions=True
model_path = "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/best_model/best_llama_seed123_roberta.pth"  # Update if different
pretrained_model = "roberta-large"              # As used during training
tokenizer = RobertaTokenizer.from_pretrained(pretrained_model)

# Load model with attention enabled
model = RobertaForSequenceClassification.from_pretrained(pretrained_model, num_labels=3, output_attentions=True)
model = resize_position_embeddings(model, new_max_pos=1024)
checkpoint = torch.load(model_path, map_location=torch.device('cpu'), weights_only=False)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Input text
text = "Vaccines are effective in preventing severe illness and death from COVID-19."

# Tokenize input
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

# Forward pass to get outputs including attentions
with torch.no_grad():
    outputs = model(**inputs)

# Extract attention scores from all layers
all_layer_attentions = outputs.attentions  # List of 24 tensors (one per layer if using roberta-large)

# Example: Get the last layer's attention for the first head
last_layer_attention = all_layer_attentions[-1]  # Shape: (1, num_heads, seq_len, seq_len)

# Mean over heads for better interpretability
mean_attention = last_layer_attention.mean(dim=1)  # Shape: (1, seq_len, seq_len)

# Convert token IDs back to words for inspection
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Print token-wise attention to [CLS] (position 0)
print("Attention of each token to [CLS]:")
for i, token in enumerate(tokens):
    print(f"{token:15s} -> {mean_attention[0, i, 0].item():.4f}")


In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import RobertaTokenizerFast


def resize_position_embeddings(model, new_max_pos=1024):
    current_max_pos, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > current_max_pos:
        new_pos_embed = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos_embed.weight.data[:current_max_pos, :] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos_embed.weight.data[current_max_pos:, :] = model.roberta.embeddings.position_embeddings.weight.data[-1, :].repeat(new_max_pos - current_max_pos, 1)
        model.roberta.embeddings.position_embeddings = new_pos_embed
        model.config.max_position_embeddings = new_max_pos
    return model

# Load tokenizer and model
pretrained_model = "roberta-large"
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(pretrained_model, num_labels=3, output_attentions=True)
model = resize_position_embeddings(model, new_max_pos=1024)

# Load trained model
model_path = "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/best_model/best_llama_seed123_roberta.pth"
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Input text
text = """he claim suggests that Fintan O’Toole wrote a column in The Irish Times expressing pity
for the United States due to President Trump’s handling of the COVID-19 pandemic. This
can be verified through multiple sources. Firstly, Fintan O’Toole is a well-known columnist
for The Irish Times, and his opinions are widely respected. Secondly, President Trump’s
leadership during the pandemic was widely criticized globally, including by many in Ireland.
It is plausible that O’Toole would express sympathy for the US in light of this criticism.
Furthermore, The Irish Times has a reputation for publishing high-quality journalism, and it
is unlikely that they would publish a column without fact-checking its content. Therefore, it
is reasonable to conclude that the claim is accurate. The combination of O’Toole’s credibility
as a columnist, the global criticism of Trump’s leadership, and The Irish Times’ reputation
for quality journalism all support the validity of the claim"""

# Tokenize with offset mappings
encoding = tokenizer(text, return_tensors="pt", return_offsets_mapping=True, truncation=True, padding=True, max_length=512)
offset_mapping = encoding.pop("offset_mapping")[0]
input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]

# Get attention
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    last_layer_attention = outputs.attentions[-1]  # Shape: (1, heads, seq_len, seq_len)
    mean_attention = last_layer_attention.mean(dim=1)[0]  # Shape: (seq_len, seq_len)

# Map subword tokens to original words
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
cls_attention = mean_attention[:, 0]  # attention of all tokens to [CLS]

# Group attention scores by original word using offset mapping
words = []
word_attentions = []
current_word = ""
current_attention = []
last_end = -1

for i, (token, offset) in enumerate(zip(tokens, offset_mapping)):
    if token in ["<s>", "</s>"]:
        continue
    start, end = offset.tolist()
    if start == last_end:  # continuation of previous word (subword)
        current_word += token.replace("Ġ", "")
        current_attention.append(cls_attention[i].item())
    else:  # new word
        if current_word:
            words.append(current_word)
            word_attentions.append(sum(current_attention) / len(current_attention))
        current_word = token.replace("Ġ", "")
        current_attention = [cls_attention[i].item()]
    last_end = end

# Append the final word
if current_word and current_attention:
    words.append(current_word)
    word_attentions.append(sum(current_attention) / len(current_attention))

# Print final wordwise attention
print("\nWordwise Attention to [CLS]:")
for word, score in zip(words, word_attentions):
    print(f"{word:15s} -> {score:.4f}")


In [ ]:
print(word_attentions)

### code for attention, working code 

In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import RobertaTokenizerFast


def resize_position_embeddings(model, new_max_pos=1024):
    current_max_pos, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > current_max_pos:
        new_pos_embed = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos_embed.weight.data[:current_max_pos, :] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos_embed.weight.data[current_max_pos:, :] = model.roberta.embeddings.position_embeddings.weight.data[-1, :].repeat(new_max_pos - current_max_pos, 1)
        model.roberta.embeddings.position_embeddings = new_pos_embed
        model.config.max_position_embeddings = new_max_pos
    return model

# Load tokenizer and model
pretrained_model = "roberta-large"
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(pretrained_model, num_labels=3, output_attentions=True)
model = resize_position_embeddings(model, new_max_pos=1024)

# Load trained model
model_path = "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/best_model/best_llama_seed123_roberta.pth"
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Input text
text = """he claim suggests that Fintan O’Toole wrote a column in The Irish Times expressing pity
for the United States due to President Trump’s handling of the COVID-19 pandemic. This
can be verified through multiple sources. Firstly, Fintan O’Toole is a well-known columnist
for The Irish Times, and his opinions are widely respected. Secondly, President Trump’s
leadership during the pandemic was widely criticized globally, including by many in Ireland.
It is plausible that O’Toole would express sympathy for the US in light of this criticism.
Furthermore, The Irish Times has a reputation for publishing high-quality journalism, and it
is unlikely that they would publish a column without fact-checking its content. Therefore, it
is reasonable to conclude that the claim is accurate. The combination of O’Toole’s credibility
as a columnist, the global criticism of Trump’s leadership, and The Irish Times’ reputation
for quality journalism all support the validity of the claim"""

# Tokenize with offset mappings
encoding = tokenizer(text, return_tensors="pt", return_offsets_mapping=True, truncation=True, padding=True, max_length=512)
offset_mapping = encoding.pop("offset_mapping")[0]
input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]

# Get attention
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    last_layer_attention = outputs.attentions[-1]  # Shape: (1, heads, seq_len, seq_len)
    mean_attention = last_layer_attention.mean(dim=1)[0]  # Shape: (seq_len, seq_len)

# Map subword tokens to original words
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
cls_attention = mean_attention[:, 0]  # attention of all tokens to [CLS]

# Group attention scores by original word using offset mapping
words = []
word_attentions = []
current_word = ""
current_attention = []
last_end = -1

for i, (token, offset) in enumerate(zip(tokens, offset_mapping)):
    if token in ["<s>", "</s>"]:
        continue
    start, end = offset.tolist()
    if start == last_end:  # continuation of previous word (subword)
        current_word += token.replace("Ġ", "")
        current_attention.append(cls_attention[i].item())
    else:  # new word
        if current_word:
            words.append(current_word)
            word_attentions.append(sum(current_attention) / len(current_attention))
        current_word = token.replace("Ġ", "")
        current_attention = [cls_attention[i].item()]
    last_end = end

# Append the final word
if current_word and current_attention:
    words.append(current_word)
    word_attentions.append(sum(current_attention) / len(current_attention))

# Print final wordwise attention
# print("\nWordwise Attention to [CLS]:")
# for word, score in zip(words, word_attentions):
#     print(f"{word:15s} -> {score:.4f}")


In [ ]:
def get_color_for_score(score):
    color_value = int((1 - score) * 100)
    return f'hwb(200 {color_value}% {20}%)'

def create_attention_heatmap_html(sentence, attention_scores, output_filename='heatmap.html'):
    # Validate the input to make sure the attention scores match the number of words in the sentence

    # Normalize the attention scores between 0 and 1
    min_score = min(attention_scores)
    max_score = max(attention_scores)
    attention_scores = [(score - min_score) / (max_score - min_score) for score in attention_scores]

    words = sentence.split(" ")
    if len(words) != len(attention_scores):
        print(f"Number of words: {len(words)}, Number of attention scores: {len(attention_scores)}")
        raise ValueError("The number of attention scores must match the number of words in the sentence.")
    
    html_content = """
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8">
            <title>Attention Heatmap</title>
            <style>
                body {
                    font-family: Arial, sans-serif;
                    font-size: 10px;  /* Smaller font */
                    margin: 10px;
                    max-width: 700px; /* Constrain width to avoid overflow */
                }
                .word {
                    display: inline-block;
                    font-weight: normal;
                    padding: 2px 4px;  /* Smaller padding */
                    margin: 1px;
                    border-radius: 3px;
                    word-break: break-word;  /* Wrap long tokens */
                }
            </style>
        </head>
        <body>
        <div>
        """
    
    # Iterate over words and attention scores to generate HTML with color-coded background
    for i, word in enumerate(words):
        score = attention_scores[i]
        color_code = get_color_for_score(score)
        html_content += f'<span class="word" style="background-color: {color_code}; color: {"white" if score > 0.5 else "black"};">{word}</span>'
    
    # Closing HTML tags
    html_content += """
    </div>
    </body>
    </html>
    """
    
    # Return the generated HTML content (for display purposes or writing to file)
    return html_content



In [ ]:
paragraph = ' '.join(words)

In [ ]:
html_content = create_attention_heatmap_html(paragraph, word_attentions)
from IPython.display import HTML, display

display(HTML(html_content))

In [ ]:
from weasyprint import HTML

with open("attention_heatmap.html", "w", encoding="utf-8") as f:
    f.write(html_content)

HTML("attention_heatmap.html").write_pdf("attention_heatmap.pdf")



## experiment for the better presentation

In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import RobertaTokenizerFast


def resize_position_embeddings(model, new_max_pos=1024):
    current_max_pos, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > current_max_pos:
        new_pos_embed = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos_embed.weight.data[:current_max_pos, :] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos_embed.weight.data[current_max_pos:, :] = model.roberta.embeddings.position_embeddings.weight.data[-1, :].repeat(new_max_pos - current_max_pos, 1)
        model.roberta.embeddings.position_embeddings = new_pos_embed
        model.config.max_position_embeddings = new_max_pos
    return model

# Load tokenizer and model
pretrained_model = "roberta-large"
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(pretrained_model, num_labels=3, output_attentions=True)
model = resize_position_embeddings(model, new_max_pos=1024)

# Load trained model
model_path = "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/best_model/best_llama_seed123_roberta.pth"
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Input text
text = """he claim suggests that Fintan O’Toole wrote a column in The Irish Times expressing pity
for the United States due to President Trump’s handling of the COVID-19 pandemic. This
can be verified through multiple sources. Firstly, Fintan O’Toole is a well-known columnist
for The Irish Times, and his opinions are widely respected. Secondly, President Trump’s
leadership during the pandemic was widely criticized globally, including by many in Ireland.
It is plausible that O’Toole would express sympathy for the US in light of this criticism.
Furthermore, The Irish Times has a reputation for publishing high-quality journalism, and it
is unlikely that they would publish a column without fact-checking its content. Therefore, it
is reasonable to conclude that the claim is accurate. The combination of O’Toole’s credibility
as a columnist, the global criticism of Trump’s leadership, and The Irish Times’ reputation
for quality journalism all support the validity of the claim"""

# Tokenize with offset mappings
encoding = tokenizer(text, return_tensors="pt", return_offsets_mapping=True, truncation=True, padding=True, max_length=512)
offset_mapping = encoding.pop("offset_mapping")[0]
input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]

# Get attention
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    last_layer_attention = outputs.attentions[-1]  # Shape: (1, heads, seq_len, seq_len)
    mean_attention = last_layer_attention.mean(dim=1)[0]  # Shape: (seq_len, seq_len)

# Map subword tokens to original words
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
cls_attention = mean_attention[:, 0]  # attention of all tokens to [CLS]

# Group attention scores by original word using offset mapping
words = []
word_attentions = []
current_word = ""
current_attention = []
last_end = -1

for i, (token, offset) in enumerate(zip(tokens, offset_mapping)):
    if token in ["<s>", "</s>"]:
        continue
    start, end = offset.tolist()
    if start == last_end:  # continuation of previous word (subword)
        current_word += token.replace("Ġ", "")
        current_attention.append(cls_attention[i].item())
    else:  # new word
        if current_word:
            words.append(current_word)
            word_attentions.append(sum(current_attention) / len(current_attention))
        current_word = token.replace("Ġ", "")
        current_attention = [cls_attention[i].item()]
    last_end = end

# Append the final word
if current_word and current_attention:
    words.append(current_word)
    word_attentions.append(sum(current_attention) / len(current_attention))

In [ ]:
def get_color_for_score(score):
    color_value = int((1 - score) * 100)
    return f'hwb(200 {color_value}% {20}%)'

def create_attention_heatmap_html(sentence, attention_scores, output_filename='heatmap.html'):
    # Validate the input to make sure the attention scores match the number of words in the sentence

    # Normalize the attention scores between 0 and 1
    min_score = min(attention_scores)
    max_score = max(attention_scores)
    attention_scores = [(score - min_score) / (max_score - min_score) for score in attention_scores]

    words = sentence.split(" ")
    if len(words) != len(attention_scores):
        print(f"Number of words: {len(words)}, Number of attention scores: {len(attention_scores)}")
        raise ValueError("The number of attention scores must match the number of words in the sentence.")
    
    html_content = """
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8">
            <title>Attention Heatmap</title>
            <style>
                body {
                    font-family: Arial, sans-serif;
                    font-size: 10px;  /* Smaller font */
                    margin: 10px;
                    max-width: 700px; /* Constrain width to avoid overflow */
                }
                .word {
                    display: inline-block;
                    font-weight: normal;
                    padding: 2px 4px;  /* Smaller padding */
                    margin: 1px;
                    border-radius: 3px;
                    word-break: break-word;  /* Wrap long tokens */
                }
            </style>
        </head>
        <body>
        <div>
        """
    
    # Iterate over words and attention scores to generate HTML with color-coded background
    for i, word in enumerate(words):
        score = attention_scores[i]
        color_code = get_color_for_score(score)
        html_content += f'<span class="word" style="background-color: {color_code}; color: {"white" if score > 0.5 else "black"};">{word}</span>'
    
    # Closing HTML tags
    html_content += """
    </div>
    </body>
    </html>
    """
    
    # Return the generated HTML content (for display purposes or writing to file)
    return html_content
paragraph = ' '.join(words)

In [ ]:
html_content = create_attention_heatmap_html(paragraph, word_attentions)
from IPython.display import HTML, display

display(HTML(html_content))

In [ ]:
def create_color_scale(th, mx):
    """
    Create a color scale div that will show the range of attention scores
    from 0 to 1 with a gradient and labels, including CSS for styling.
    """
    color_scale_html = f"""
    <style>
        .color-scale {{
            width: 100%;
            height: 20px;
            background: linear-gradient(to right, hwb(200 0% 20%), hwb(200 100% 20%));
            margin-top: 20px;
        }}
        .scale-labels {{
            display: flex;
            justify-content: space-between;
            font-size: 14px;
        }}
    </style>
    <div class="color-scale"></div>
    <div class="scale-labels">
        <span>{mx}</span>
        <span>{th}</span>
    </div>
    """
    return color_scale_html

color_scale = create_color_scale(0.6, 0.9)

# Display the color scale
from IPython.display import display, HTML
display(HTML(color_scale))

In [ ]:
!sudo apt install wkhtmltopdf

In [ ]:
import torch
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from weasyprint import HTML

# 1) Utility: Resize RoBERTa positional embeddings (so they match your trained model)
def resize_position_embeddings(model, new_max_pos=1024):
    curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > curr_max:
        new_pos_embed = torch.nn.Embedding(new_max_pos, embed_size)
        # copy old + pad
        new_pos_embed.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos_embed.weight.data[curr_max:] = model.roberta.embeddings.position_embeddings.weight.data[-1].unsqueeze(0).repeat(new_max_pos - curr_max, 1)
        model.roberta.embeddings.position_embeddings = new_pos_embed
        model.config.max_position_embeddings = new_max_pos
    return model

# 2) Compute word-level attention scores to [CLS]
def get_word_attentions(model, tokenizer, text, max_length=512):
    enc = tokenizer(text,
                    return_tensors="pt",
                    return_offsets_mapping=True,
                    truncation=True,
                    padding="max_length",
                    max_length=max_length)
    offsets = enc.pop("offset_mapping")[0]
    input_ids = enc["input_ids"]
    mask      = enc["attention_mask"]

    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=mask)
        attn = out.attentions[-1]          # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]    # (L, L)
    cls_attn = mean_attn[:, 0]            # (L,)

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    words, word_scores = [], []
    cur_word, cur_scores, last_end = "", [], -1

    for tok, (start, end), score in zip(tokens, offsets.tolist(), cls_attn.tolist()):
        # skip special/pad tokens
        if tok in tokenizer.all_special_tokens:
            continue

        # strip the Ġ marker
        piece = tok.replace("Ġ", "")

        if start == last_end:  # continuation
            cur_word   += piece
            cur_scores.append(score)
        else:                  # new word
            if cur_word:  # flush previous
                words.append(cur_word)
                word_scores.append(sum(cur_scores)/len(cur_scores))
            cur_word, cur_scores = piece, [score]
        last_end = end

    # flush last
    if cur_word:
        words.append(cur_word)
        word_scores.append(sum(cur_scores)/len(cur_scores))

    return words, word_scores

# 3) Color mapping (hwb)
def get_color(score):
    pct = (1-score)*100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Section texts (paste your exact extracted strings here)
claim_text = (
    "The Irish Times newspaper published a column by Fintan O’Toole expressing "
    "\"pity\" for the United States over U.S. President Donald Trump’s leadership during the COVID-19 pandemic."
)

true_text = (
    "The claim suggests that Fintan O’Toole wrote a column in The Irish Times expressing pity for the "
    "United States due to President Trump’s handling of the COVID-19 pandemic. This can be verified through "
    "multiple sources. Firstly, Fintan O’Toole is a well-known columnist for The Irish Times, and his opinions "
    "are widely respected. Secondly, President Trump’s leadership during the pandemic was widely criticized globally, "
    "including by many in Ireland. It is plausible that O’Toole would express sympathy for the US in light of this "
    "criticism. Furthermore, The Irish Times has a reputation for publishing high-quality journalism, and it is unlikely "
    "that they would publish a column without fact-checking its content. Therefore, it is reasonable to conclude that "
    "the claim is accurate. The combination of O’Toole’s credibility as a columnist, the global criticism of Trump’s "
    "leadership, and The Irish Times’ reputation for quality journalism all support the validity of the claim."
)

false_text = (
    "The claim that The Irish Times newspaper published a column by Fintan O’Toole expressing \"pity\" for the "
    "United States over U.S. President Donald Trump’s leadership during the COVID-19 pandemic is challenged by the "
    "content of the column itself. While O’Toole does express sympathy for the majority of Americans who did not vote "
    "for Trump, he does not convey a sense of pity for the country as a whole. In fact, he suggests that the United "
    "States has historically evoked a wide range of emotions globally, including love, hatred, fear, and contempt. "
    "Furthermore, O’Toole critiques Trump’s leadership, calling him an \"authoritarian and con man,\" and criticizes "
    "the president’s handling of the pandemic, stating that he is \"actively promoting the spread of a fatal disease.\" "
    "The tone of the column is critical of Trump and his administration, rather than expressing pity for the country. "
    "Therefore, the claim appears to be inaccurate."
)

# 5) Load tokenizer & model
pretrained = "roberta-large"
tokenizer   = RobertaTokenizerFast.from_pretrained(pretrained)
model       = RobertaForSequenceClassification.from_pretrained(pretrained,
                                                               num_labels=3,
                                                               output_attentions=True)
model       = resize_position_embeddings(model, new_max_pos=1024)

# point to your .pth file here
checkpoint = torch.load("/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/best_model/best_llama_seed123_roberta.pth",
                        map_location="cpu",
                        weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# 6) Compute for each section
sections = {
    "Claim":                claim_text,
    "True Justification":   true_text,
    "False Justification":  false_text,
}
results = {}
for title, txt in sections.items():
    words, scores = get_word_attentions(model, tokenizer, txt)
    # normalize per-section
    mn, mx = min(scores), max(scores)
    norm = [(s-mn)/(mx-mn+1e-12) for s in scores]
    results[title] = (words, norm)

# 7) Build compact HTML
html = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Heatmap</title>
  <style>
    body {
      font-family: Arial, sans-serif;
      font-size: 10px;
      margin: 10px;
      max-width: 700px;
    }
    .section-title {
      font-size: 12px;
      font-weight: bold;
      margin: 8px 0 4px;
    }
    .word {
      display: inline-block;
      padding: 2px 4px;
      margin: 1px;
      border-radius: 3px;
      word-break: break-word;
    }
  </style>
</head>
<body>
"""

for title, (words, norm_scores) in results.items():
    html += f'<div class="section-title">{title}</div>\n'
    for w, sc in zip(words, norm_scores):
        color = get_color(sc)
        txtcol = "white" if sc>0.5 else "black"
        html += f'<span class="word" style="background:{color}; color:{txtcol}">{w}</span>'
    html += "\n<br/>\n"

html += "</body></html>"

# 8) Write HTML + PDF
with open("attention_heatmap.html", "w", encoding="utf-8") as f:
    f.write(html)

HTML("attention_heatmap.html").write_pdf("attention_heatmap.pdf")

print("✅ Generated: attention_heatmap.html & attention_heatmap.pdf")


In [ ]:
import torch
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from weasyprint import HTML

# 1) Resize positional embeddings so they match your trained 1024-pos model
def resize_position_embeddings(model, new_max_pos=1024):
    curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > curr_max:
        new_pos = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos.weight.data[curr_max:] = (
            model.roberta.embeddings.position_embeddings.weight.data[-1]
            .unsqueeze(0)
            .repeat(new_max_pos - curr_max, 1)
        )
        model.roberta.embeddings.position_embeddings = new_pos
        model.config.max_position_embeddings = new_max_pos
    return model

# 2) Compute word-level attention for a single text
def get_word_attentions(model, tokenizer, text):
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    offsets = enc.pop("offset_mapping")[0]
    ids, mask = enc["input_ids"], enc["attention_mask"]

    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask)
        attn = out.attentions[-1]             # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]       # (L, L)
    cls_attn = mean_attn[:, 0]               # (L,)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])
    words, scores = [], []
    cur_word, cur_scores, last_end = "", [], -1

    for tok, (start, end), score in zip(tokens, offsets.tolist(), cls_attn.tolist()):
        if tok in tokenizer.all_special_tokens:
            continue
        piece = tok.replace("Ġ", "")
        if start == last_end:
            cur_word += piece
            cur_scores.append(score)
        else:
            if cur_word:
                words.append(cur_word)
                scores.append(sum(cur_scores)/len(cur_scores))
            cur_word, cur_scores = piece, [score]
        last_end = end

    if cur_word:
        words.append(cur_word)
        scores.append(sum(cur_scores)/len(cur_scores))

    return words, scores

# 3) Map normalized score → color
def get_color(score):
    pct = (1 - score) * 100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Your three text segments (exactly as you want them printed)
claim_text = (
    "52 thoroughbred horses whose owner died from COVID-19 are destined for slaughter if adopters are not found for them soon."
)
true_text = (
    "The claim is likely true. The provided evidence suggests that there is an urgent need to find new homes for 52 "
    "thoroughbred horses due to the death of their owner. This implies that the horses are indeed at risk of being sent "
    "to slaughter if adopters are not found soon. The fact that a circulating message is asking recipients to help find "
    "new homes for the horses further supports the urgency of the situation. Additionally, the specific number of horses "
    "mentioned (52) and the breed (thoroughbred) provide context and credibility to the claim. Overall, the evidence "
    "suggests that the claim is based on a genuine concern for the welfare of the horses and is not a fabricated or exaggerated story."
)
false_text = (
    "The claim that 52 thoroughbred horses are destined for slaughter if adopters are not found for them soon is challenged "
    "by several key factors. Firstly, the claim has been circulating for several years, with the original post dating back to 2011. "
    "According to credible reports on horse-related forums, all 52 horses were actually rehomed, with most going to family friends "
    "of the deceased owner. This suggests that the claim is not based on current events, but rather a recycled and outdated story. "
    "Furthermore, the claim has undergone subtle changes over the years, including the reason for the horses being in danger. "
    "Initially, the post stated that the horses would be sent to a glue factory, but more recently, the reason cited is the owner's passing due to COVID-19. "
    "However, there is no evidence to suggest that the horses are currently in danger or that they are being considered for slaughter. "
    "In fact, according to TheHorse.com, all 52 horses were able to find homes within a week of the initial post in 2011. "
    "This contradicts the claim that the horses are in imminent danger of being slaughtered. "
    "Therefore, based on the available evidence, it appears that the claim is not supported by facts and is likely a recycled and outdated story."
)

gold_label, pred_label = "false", "false"

# 5) Load tokenizer & model
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large", num_labels=3, output_attentions=True
)
model = resize_position_embeddings(model, new_max_pos=1024)
ckpt = torch.load(
    "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/best_model/best_llama_seed123_roberta.pth",
    map_location="cpu",
    weights_only=False,
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 6) Concatenate with separators exactly as during training
sep      = tokenizer.sep_token
combined = f"{claim_text} {sep} {true_text} {sep} {false_text}"

# 7) Extract attentions
all_words, all_scores = get_word_attentions(model, tokenizer, combined)

# 8) Split back into counts
cw, _ = get_word_attentions(model, tokenizer, claim_text)
tw, _ = get_word_attentions(model, tokenizer, true_text)
fw, _ = get_word_attentions(model, tokenizer, false_text)
c_n, t_n, f_n = len(cw), len(tw), len(fw)

sections = {
    "True Justifications":   (all_words[c_n:c_n+t_n],  all_scores[c_n:c_n+t_n]),
    "False Justifications":  (all_words[c_n+t_n:],     all_scores[c_n+t_n:]),
}

# 9) Build the HTML in the exact layout you showed
html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Sample</title>
  <style>
    body {{ font-family: Arial; font-size:11px; margin:20px; }}
    .border {{ border-top:1px solid #000; border-bottom:1px solid #000; padding:10px 0; }}
    .claim, .label, .section-title {{ margin:8px 0; }}
    .claim strong, .label strong {{ font-size:12px; }}
    .word {{ display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }}
  </style>
</head>
<body>

<div class="border">
  <p class="claim"><strong>Claim:</strong> “{claim_text}”</p>
  <p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>
  <p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>
</div>

"""

# 10) For each justification section, print header + coloured spans
for title, (words, scores) in sections.items():
    # threshold for top 25%
    thr = np.percentile(scores, 75)
    mn, mx = min(scores), max(scores)
    html += f'<p class="section-title"><strong>{title}:</strong></p>\n'
    html += '<p>“'
    for w, sc in zip(words, scores):
        norm = (sc - mn) / (mx - mn + 1e-12)
        if sc >= thr:
            color = get_color(norm)
            fg    = "white" if norm > 0.5 else "black"
            html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
        else:
            html += f"{w} "
    html += '”</p>\n\n'

# 11) Close & Write PDF
html += "</body></html>"

with open("attention_sample.html","w",encoding="utf-8") as f:
    f.write(html)

HTML("attention_sample.html").write_pdf("attention_sample.pdf")

print("✅ attention_sample.html & attention_sample.pdf generated.")


## output added in the paper

In [ ]:
import torch
import re
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from weasyprint import HTML
import string
SEP = tokenizer.sep_token
# 1) Resize positional embeddings
def resize_position_embeddings(model, new_max_pos=1024):
    curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > curr_max:
        new_pos = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos.weight.data[curr_max:] = (
            model.roberta.embeddings.position_embeddings.weight.data[-1]
            .unsqueeze(0)
            .repeat(new_max_pos - curr_max, 1)
        )
        model.roberta.embeddings.position_embeddings = new_pos
        model.config.max_position_embeddings = new_max_pos
    return model

# 2) Word-level attention extraction
def get_word_attentions_with_spans(model, tokenizer, text):
    """
    Returns three lists:
     - words: the exact substrings from `text`
     - scores: the average [CLS]-attention for each substring
     - spans:  the (start, end) character indices in `text`
    Filters out any pure separators or punctuation spans.
    """
    SEP = tokenizer.sep_token  # e.g. "</s>"

    # Tokenize and get offsets
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    ids, mask = enc["input_ids"], enc["attention_mask"]

    # Forward pass to get attentions
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask)
        attn = out.attentions[-1]           # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]     # (L, L)
    cls_attn = mean_attn[:, 0].tolist()     # (L,)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])

    words, scores, spans = [], [], []
    last_end = -1
    grp_start = grp_end = None
    grp_scores = []

    for tok, (start, end), score in zip(tokens, offsets, cls_attn):
        # Skip special & pad tokens
        if tok in tokenizer.all_special_tokens:
            continue

        # If this token does not continue the previous span → flush old group
        if start != last_end:
            if grp_start is not None:
                substring = text[grp_start:grp_end]
                # filter out separator or angle-bracket artifacts
                if SEP in substring or '<' in substring or '>' in substring:
                    pass
                # only keep if contains at least one alphanumeric
                elif re.search(r'\w', substring):
                    words.append(substring)
                    scores.append(sum(grp_scores) / len(grp_scores))
                    spans.append((grp_start, grp_end))
            # start new group
            grp_start, grp_end = start, end
            grp_scores = [score]
        else:
            # continuation of same word
            grp_end = end
            grp_scores.append(score)

        last_end = end

    # flush final group
    if grp_start is not None:
        substring = text[grp_start:grp_end]
        if not (SEP in substring or '<' in substring or '>' in substring) and re.search(r'\w', substring):
            words.append(substring)
            scores.append(sum(grp_scores) / len(grp_scores))
            spans.append((grp_start, grp_end))

    return words, scores, spans

# 3) Color mapping
def get_color(score):
    pct = (1 - score) * 100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Texts & labels
claim_text = (
    """The Irish Times newspaper published a column by Fintan O’Toole expressing
\"pity\" for the United States over U.S. President Donald Trump’s leadership during the
COVID-19 pandemic"""
)
true_text = (
    """The claim suggests that Fintan O’Toole wrote a column in The Irish Times expressing pity
for the United States due to President Trump’s handling of the COVID-19 pandemic. This
can be verified through multiple sources. Firstly, Fintan O’Toole is a well-known columnist
for The Irish Times, and his opinions are widely respected. Secondly, President Trump’s
leadership during the pandemic was widely criticized globally, including by many in Ireland.
It is plausible that O’Toole would express sympathy for the US in light of this criticism.
Furthermore, The Irish Times has a reputation for publishing high-quality journalism, and it
is unlikely that they would publish a column without fact-checking its content. Therefore, it
is reasonable to conclude that the claim is accurate. The combination of O’Toole’s credibility
as a columnist, the global criticism of Trump’s leadership, and The Irish Times’ reputation
for quality journalism all support the validity of the claim"""
)
false_text = (
    """The claim that The Irish Times newspaper published a column by Fintan O’Toole expressing
\"pity\" for the United States over U.S. President Donald Trump’s leadership during the
COVID-19 pandemic is challenged by the content of the column itself. While O’Toole does
express sympathy for the majority of Americans who did not vote for Trump, he does not
convey a sense of pity for the country as a whole. In fact, he suggests that the United States
has historically evoked a wide range of emotions globally, including love, hatred, fear, and
contempt. Furthermore, O’Toole critiques Trump’s leadership, calling him an \"authoritarian
and con man,\" and criticizes the president’s handling of the pandemic, stating that he is
\"actively promoting the spread of a fatal disease.\" The tone of the column is critical of
Trump and his administration, rather than expressing pity for the country. Therefore, the
claim appears to be inaccurate"""
)

gold_label, pred_label = "true", "true"

# 5) Load model & tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large", num_labels=3, output_attentions=True
)
model = resize_position_embeddings(model, new_max_pos=1024)
ckpt = torch.load(
    "best_model/best_llama_seed123_roberta.pth",
    map_location="cpu",
    weights_only=False,
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 6) Concatenate & extract from combined
sep       = tokenizer.sep_token
combined  = f"{claim_text} {sep} {true_text} {sep} {false_text}"
words, scores, spans = get_word_attentions_with_spans(model, tokenizer, combined)

# find sep positions in the combined string
i1 = combined.find(sep)
i2 = combined.find(sep, i1 + len(sep))
end1 = i1 + len(sep)
end2 = i2 + len(sep)

# 7) Split into three lists by span
claim_words,  claim_scores  = [], []
true_words,   true_scores   = [], []
false_words,  false_scores  = [], []

for w, sc, (st, en) in zip(words, scores, spans):
    if en <= i1:
        claim_words.append(w);  claim_scores.append(sc)
    elif st >= end1 and en <= i2:
        true_words.append(w);   true_scores.append(sc)
    elif st >= end2:
        false_words.append(w);  false_scores.append(sc)
    # else: it’s part of the separator; skip

sections = {
    "Claim":               (claim_words,  claim_scores),
    "Support Justifications": (true_words,  true_scores),
    "Refute Justifications":(false_words, false_scores),
}


# 7) Build HTML (no borders, all sections highlighted)
html = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Sample</title>
  <style>
    body { font-family: Arial; font-size:11px; margin:20px; }
    .section-title { margin:12px 0 4px; font-weight:bold; }
    .label { margin:4px 0; }
    .word { display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }
  </style>
</head>
<body>
"""

# Add Claim & labels
html += f'<p class="section-title">Claim:</p>\n<p>“'
# highlight top-25% in claim
cw_words, cw_scores = sections["Claim"]
thr_c = np.percentile(cw_scores, 75)
mn_c, mx_c = min(cw_scores), max(cw_scores)
for w, sc in zip(cw_words, cw_scores):
    norm = (sc - mn_c) / (mx_c - mn_c + 1e-12)
    if sc >= thr_c:
        color = get_color(norm)
        fg    = "white" if norm>0.5 else "black"
        html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
    else:
        html += f"{w} "
html += '”</p>\n'

html += f'<p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>\n'
html += f'<p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>\n'

# Add True/False Justifications
for title in ["Support Justifications", "Refute Justifications"]:
    words, scores = sections[title]
    thr = np.percentile(scores, 75)
    mn, mx = min(scores), max(scores)

    html += f'<p class="section-title">{title}:</p>\n<p>“'
    for w, sc in zip(words, scores):
        norm = (sc - mn) / (mx - mn + 1e-12)
        if sc >= thr:
            color = get_color(norm)
            fg    = "white" if norm>0.5 else "black"
            html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
        else:
            html += f"{w} "
    html += '”</p>\n'

html += "</body></html>"

# 8) Save HTML + PDF
with open("attention_sample.html", "w", encoding="utf-8") as f:
    f.write(html)
HTML("attention_sample.html").write_pdf("attention_sample_14.pdf")

print("✅ attention_sample.html & attention_sample_14.pdf generated.")


## example 2

In [ ]:
# import torch
# import numpy as np
# from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
# from weasyprint import HTML

# # 1) Resize positional embeddings
# def resize_position_embeddings(model, new_max_pos=1024):
#     curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
#     if new_max_pos > curr_max:
#         new_pos = torch.nn.Embedding(new_max_pos, embed_size)
#         new_pos.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
#         new_pos.weight.data[curr_max:] = (
#             model.roberta.embeddings.position_embeddings.weight.data[-1]
#             .unsqueeze(0)
#             .repeat(new_max_pos - curr_max, 1)
#         )
#         model.roberta.embeddings.position_embeddings = new_pos
#         model.config.max_position_embeddings = new_max_pos
#     return model

# # 2) Word-level attention extraction
# def get_word_attentions(model, tokenizer, text):
#     enc = tokenizer(
#         text,
#         return_tensors="pt",
#         return_offsets_mapping=True,
#         truncation=True,
#         padding="max_length",
#         max_length=512,
#     )
#     offsets = enc.pop("offset_mapping")[0]
#     ids, mask = enc["input_ids"], enc["attention_mask"]

#     with torch.no_grad():
#         out = model(input_ids=ids, attention_mask=mask)
#         attn = out.attentions[-1]             # (1, heads, L, L)
#         mean_attn = attn.mean(dim=1)[0]       # (L, L)
#     cls_attn = mean_attn[:, 0]               # (L,)

#     tokens = tokenizer.convert_ids_to_tokens(ids[0])
#     words, scores = [], []
#     cur_word, cur_scores, last_end = "", [], -1

#     for tok, (start, end), score in zip(tokens, offsets.tolist(), cls_attn.tolist()):
#         if tok in tokenizer.all_special_tokens:
#             continue
#         piece = tok.replace("Ġ", "")
#         if start == last_end:
#             cur_word += piece
#             cur_scores.append(score)
#         else:
#             if cur_word:
#                 words.append(cur_word)
#                 scores.append(sum(cur_scores)/len(cur_scores))
#             cur_word, cur_scores = piece, [score]
#         last_end = end

#     if cur_word:
#         words.append(cur_word)
#         scores.append(sum(cur_scores)/len(cur_scores))

#     return words, scores

# # 3) Color mapping
# def get_color(score):
#     pct = (1 - score) * 100
#     return f"hwb(200 {pct:.1f}% 20%)"

# # 4) Texts & labels
# claim_text = (
#     """52 thoroughbred horses whose owner died from COVID-19 are destined for
# slaughter if adopters are not found for them soon."""
# )
# true_text = (
#     """The claim is likely true. The provided evidence suggests that there is an urgent need to find
# new homes for 52 thoroughbred horses due to the death of their owner. This implies that the
# horses are indeed at risk of being sent to slaughter if adopters are not found soon. The fact
# that a circulating message is asking recipients to help find new homes for the horses further
# supports the urgency of the situation. Additionally, the specific number of horses mentioned
# (52) and the breed (thoroughbred) provide context and credibility to the claim. Overall, the
# evidence suggests that the claim is based on a genuine concern for the welfare of the horses
# and is not a fabricated or exaggerated story"""
# )
# false_text = (
#     """The claim that 52 thoroughbred horses are destined for slaughter if adopters are not found
# for them soon is challenged by several key factors. Firstly, the claim has been circulating
# for several years, with the original post dating back to 2011. According to credible reports
# on horse-related forums, all 52 horses were actually rehomed, with most going to family
# friends of the deceased owner. This suggests that the claim is not based on current events, but
# rather a recycled and outdated story. Furthermore, the claim has undergone subtle changes
# over the years, including the reason for the horses being in danger. Initially, the post stated
# that the horses would be sent to a glue factory, but more recently, the reason cited is the
# owner’s passing due to COVID-19. However, there is no evidence to suggest that the horses
# are currently in danger or that they are being considered for slaughter. In fact, according to
# TheHorse. Com, all 52 horses were able to find homes within a week of the initial post in
# 2011. This contradicts the claim that the horses are in imminent danger of being slaughtered.
# Therefore, based on the available evidence, it appears that the claim is not supported by facts
# and is likely a recycled and outdated story"""
# )

# gold_label, pred_label = "false", "false"

# # 5) Load model & tokenizer
# tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
# model = RobertaForSequenceClassification.from_pretrained(
#     "roberta-large", num_labels=3, output_attentions=True
# )
# model = resize_position_embeddings(model, new_max_pos=1024)
# ckpt = torch.load(
#     "best_model/best_llama_seed123_roberta.pth",
#     map_location="cpu",
#     weights_only=False,
# )
# model.load_state_dict(ckpt["model_state_dict"])
# model.eval()

# # 6) Concatenate & extract from combined
# sep      = tokenizer.sep_token
# combined = f"{claim_text} {sep} {true_text} {sep} {false_text}"
# all_w, all_s = get_word_attentions(model, tokenizer, combined)

# # split counts
# cw, _ = get_word_attentions(model, tokenizer, claim_text)
# tw, _ = get_word_attentions(model, tokenizer, true_text)
# fw, _ = get_word_attentions(model, tokenizer, false_text)
# c_n, t_n, f_n = len(cw), len(tw), len(fw)

# sections = {
#     "Claim":                (all_w[0:c_n],         all_s[0:c_n]),
#     "True Justifications":  (all_w[c_n:c_n+t_n],   all_s[c_n:c_n+t_n]),
#     "False Justifications": (all_w[c_n+t_n:],      all_s[c_n+t_n:])
# }

# # 7) Build HTML (no borders, all sections highlighted)
# html = """
# <!DOCTYPE html>
# <html lang="en">
# <head>
#   <meta charset="UTF-8">
#   <title>Attention Sample</title>
#   <style>
#     body { font-family: Arial; font-size:11px; margin:20px; }
#     .section-title { margin:12px 0 4px; font-weight:bold; }
#     .label { margin:4px 0; }
#     .word { display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }
#   </style>
# </head>
# <body>
# """

# # Add Claim & labels
# html += f'<p class="section-title">Claim:</p>\n<p>“'
# # highlight top-25% in claim
# cw_words, cw_scores = sections["Claim"]
# thr_c = np.percentile(cw_scores, 50)
# mn_c, mx_c = min(cw_scores), max(cw_scores)
# for w, sc in zip(cw_words, cw_scores):
#     norm = (sc - mn_c) / (mx_c - mn_c + 1e-12)
#     if sc >= thr_c:
#         color = get_color(norm)
#         fg    = "white" if norm>0.5 else "black"
#         html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
#     else:
#         html += f"{w} "
# html += '”</p>\n'

# html += f'<p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>\n'
# html += f'<p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>\n'

# # Add True/False Justifications
# for title in ["True Justifications", "False Justifications"]:
#     words, scores = sections[title]
#     thr = np.percentile(scores, 50)
#     mn, mx = min(scores), max(scores)

#     html += f'<p class="section-title">{title}:</p>\n<p>“'
#     for w, sc in zip(words, scores):
#         norm = (sc - mn) / (mx - mn + 1e-12)
#         if sc >= thr:
#             color = get_color(norm)
#             fg    = "white" if norm>0.5 else "black"
#             html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
#         else:
#             html += f"{w} "
#     html += '”</p>\n'

# html += "</body></html>"

# # 8) Save HTML + PDF
# with open("attention_sample.html", "w", encoding="utf-8") as f:
#     f.write(html)
# HTML("attention_sample.html").write_pdf("attention_sample.pdf")

# print("✅ attention_sample.html & attention_sample.pdf generated.")


In [ ]:
import torch
import re
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from weasyprint import HTML
import string
SEP = tokenizer.sep_token
# 1) Resize positional embeddings
def resize_position_embeddings(model, new_max_pos=1024):
    curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > curr_max:
        new_pos = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos.weight.data[curr_max:] = (
            model.roberta.embeddings.position_embeddings.weight.data[-1]
            .unsqueeze(0)
            .repeat(new_max_pos - curr_max, 1)
        )
        model.roberta.embeddings.position_embeddings = new_pos
        model.config.max_position_embeddings = new_max_pos
    return model

# 2) Word-level attention extraction
def get_word_attentions_with_spans(model, tokenizer, text):
    """
    Returns three lists:
     - words: the exact substrings from `text`
     - scores: the average [CLS]-attention for each substring
     - spans:  the (start, end) character indices in `text`
    Filters out any pure separators or punctuation spans.
    """
    SEP = tokenizer.sep_token  # e.g. "</s>"

    # Tokenize and get offsets
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    ids, mask = enc["input_ids"], enc["attention_mask"]

    # Forward pass to get attentions
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask)
        attn = out.attentions[-1]           # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]     # (L, L)
    cls_attn = mean_attn[:, 0].tolist()     # (L,)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])

    words, scores, spans = [], [], []
    last_end = -1
    grp_start = grp_end = None
    grp_scores = []

    for tok, (start, end), score in zip(tokens, offsets, cls_attn):
        # Skip special & pad tokens
        if tok in tokenizer.all_special_tokens:
            continue

        # If this token does not continue the previous span → flush old group
        if start != last_end:
            if grp_start is not None:
                substring = text[grp_start:grp_end]
                # filter out separator or angle-bracket artifacts
                if SEP in substring or '<' in substring or '>' in substring:
                    pass
                # only keep if contains at least one alphanumeric
                elif re.search(r'\w', substring):
                    words.append(substring)
                    scores.append(sum(grp_scores) / len(grp_scores))
                    spans.append((grp_start, grp_end))
            # start new group
            grp_start, grp_end = start, end
            grp_scores = [score]
        else:
            # continuation of same word
            grp_end = end
            grp_scores.append(score)

        last_end = end

    # flush final group
    if grp_start is not None:
        substring = text[grp_start:grp_end]
        if not (SEP in substring or '<' in substring or '>' in substring) and re.search(r'\w', substring):
            words.append(substring)
            scores.append(sum(grp_scores) / len(grp_scores))
            spans.append((grp_start, grp_end))

    return words, scores, spans

# 3) Color mapping
def get_color(score):
    pct = (1 - score) * 100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Texts & labels
claim_text = (
    """52 thoroughbred horses whose owner died from COVID-19 are destined for
slaughter if adopters are not found for them soon."""
)
true_text = (
    """The claim is likely true. The provided evidence suggests that there is an urgent need to find
new homes for 52 thoroughbred horses due to the death of their owner. This implies that the
horses are indeed at risk of being sent to slaughter if adopters are not found soon. The fact
that a circulating message is asking recipients to help find new homes for the horses further
supports the urgency of the situation. Additionally, the specific number of horses mentioned
(52) and the breed (thoroughbred) provide context and credibility to the claim. Overall, the
evidence suggests that the claim is based on a genuine concern for the welfare of the horses
and is not a fabricated or exaggerated story"""
)
false_text = (
    """The claim that 52 thoroughbred horses are destined for slaughter if adopters are not found
for them soon is challenged by several key factors. Firstly, the claim has been circulating
for several years, with the original post dating back to 2011. According to credible reports
on horse-related forums, all 52 horses were actually rehomed, with most going to family
friends of the deceased owner. This suggests that the claim is not based on current events, but
rather a recycled and outdated story. Furthermore, the claim has undergone subtle changes
over the years, including the reason for the horses being in danger. Initially, the post stated
that the horses would be sent to a glue factory, but more recently, the reason cited is the
owner’s passing due to COVID-19. However, there is no evidence to suggest that the horses
are currently in danger or that they are being considered for slaughter. In fact, according to
TheHorse. Com, all 52 horses were able to find homes within a week of the initial post in
2011. This contradicts the claim that the horses are in imminent danger of being slaughtered.
Therefore, based on the available evidence, it appears that the claim is not supported by facts
and is likely a recycled and outdated story"""
)

gold_label, pred_label = "false", "false"

# 5) Load model & tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large", num_labels=3, output_attentions=True
)
model = resize_position_embeddings(model, new_max_pos=1024)
ckpt = torch.load(
    "best_model/best_llama_seed123_roberta.pth",
    map_location="cpu",
    weights_only=False,
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 6) Concatenate & extract from combined
sep       = tokenizer.sep_token
combined  = f"{claim_text} {sep} {true_text} {sep} {false_text}"
words, scores, spans = get_word_attentions_with_spans(model, tokenizer, combined)

# find sep positions in the combined string
i1 = combined.find(sep)
i2 = combined.find(sep, i1 + len(sep))
end1 = i1 + len(sep)
end2 = i2 + len(sep)

# 7) Split into three lists by span
claim_words,  claim_scores  = [], []
true_words,   true_scores   = [], []
false_words,  false_scores  = [], []

for w, sc, (st, en) in zip(words, scores, spans):
    if en <= i1:
        claim_words.append(w);  claim_scores.append(sc)
    elif st >= end1 and en <= i2:
        true_words.append(w);   true_scores.append(sc)
    elif st >= end2:
        false_words.append(w);  false_scores.append(sc)
    # else: it’s part of the separator; skip

sections = {
    "Claim":               (claim_words,  claim_scores),
    "Support Justifications": (true_words,  true_scores),
    "Refute Justifications":(false_words, false_scores),
}


# 7) Build HTML (no borders, all sections highlighted)
html = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Sample</title>
  <style>
    body { font-family: Arial; font-size:11px; margin:20px; }
    .section-title { margin:12px 0 4px; font-weight:bold; }
    .label { margin:4px 0; }
    .word { display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }
  </style>
</head>
<body>
"""

# Add Claim & labels
html += f'<p class="section-title">Claim:</p>\n<p>“'
# highlight top-25% in claim
cw_words, cw_scores = sections["Claim"]
thr_c = np.percentile(cw_scores, 75)
mn_c, mx_c = min(cw_scores), max(cw_scores)
for w, sc in zip(cw_words, cw_scores):
    norm = (sc - mn_c) / (mx_c - mn_c + 1e-12)
    if sc >= thr_c:
        color = get_color(norm)
        fg    = "white" if norm>0.5 else "black"
        html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
    else:
        html += f"{w} "
html += '”</p>\n'

html += f'<p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>\n'
html += f'<p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>\n'

# Add True/False Justifications
for title in ["Support Justifications", "Refute Justifications"]:
    words, scores = sections[title]
    thr = np.percentile(scores, 75)
    mn, mx = min(scores), max(scores)

    html += f'<p class="section-title">{title}:</p>\n<p>“'
    for w, sc in zip(words, scores):
        norm = (sc - mn) / (mx - mn + 1e-12)
        if sc >= thr:
            color = get_color(norm)
            fg    = "white" if norm>0.5 else "black"
            html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
        else:
            html += f"{w} "
    html += '”</p>\n'

html += "</body></html>"

# 8) Save HTML + PDF
with open("attention_sample.html", "w", encoding="utf-8") as f:
    f.write(html)
HTML("attention_sample.html").write_pdf("attention_sample_15.pdf")

print("✅ attention_sample.html & attention_sample_15.pdf generated.")


## example 3*

In [ ]:
import torch
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from weasyprint import HTML

# 1) Resize RoBERTa positional embeddings to match training
def resize_position_embeddings(model, new_max_pos=1024):
    curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > curr_max:
        new_pos = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos.weight.data[curr_max:] = (
            model.roberta.embeddings.position_embeddings.weight.data[-1]
            .unsqueeze(0)
            .repeat(new_max_pos - curr_max, 1)
        )
        model.roberta.embeddings.position_embeddings = new_pos
        model.config.max_position_embeddings = new_max_pos
    return model

# 2) Compute word-level attention scores
def get_word_attentions(model, tokenizer, text):
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    offsets = enc.pop("offset_mapping")[0]
    ids, mask = enc["input_ids"], enc["attention_mask"]

    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask)
        attn = out.attentions[-1]             # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]       # (L, L)
    cls_attn = mean_attn[:, 0]               # (L,)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])
    words, scores = [], []
    cur_word, cur_scores, last_end = "", [], -1

    for tok, (start, end), score in zip(tokens, offsets.tolist(), cls_attn.tolist()):
        if tok in tokenizer.all_special_tokens:
            continue
        piece = tok.replace("Ġ", "")
        if start == last_end:
            cur_word += piece
            cur_scores.append(score)
        else:
            if cur_word:
                words.append(cur_word)
                scores.append(sum(cur_scores) / len(cur_scores))
            cur_word, cur_scores = piece, [score]
        last_end = end

    # flush last
    if cur_word:
        words.append(cur_word)
        scores.append(sum(cur_scores) / len(cur_scores))

    return words, scores

# 3) Color mapping for top‐attention highlights
def get_color(score):
    pct = (1 - score) * 100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Define your segments and labels
claim_text = (
    "Florida Democrats just voted to impose Sharia law on women."
)

true_text = ("""I can't help you with this request. I can’t verify the accuracy of claims about imposing Sharia law on women in Florida. Is there anything else I can help you with?  Would you like to know how to evaluate the credibility of sources or identify potential biases in information?  I can also provide general guidance on critical thinking and media literacy.  Please let me know how I can assist you further.  If you have any other questions or topics you'd like to discuss, feel free to ask!  I'm here to help.  Is there anything else I can help you with?  Would you like to know more about fact-checking or evaluating evidence?  I can also provide information on how to identify reliable sources and avoid misinformation.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about critical thinking or how to evaluate information?  I can also provide information on how to identify potential biases and propaganda.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about how to evaluate the credibility of sources or identify potential biases in information?  I can also provide general guidance on critical thinking and media literacy.  Please let me know how I can assist you further.  If you have any other questions or topics you'd like to discuss, feel free to ask!  I'm here to help.  Is there anything else I can help you with?  Would you like to know more about fact-checking or evaluating evidence?  I can also provide information on how to identify reliable sources and avoid misinformation.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about critical thinking or how to evaluate information?  I can also provide information on how to identify potential biases and propaganda.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about how to evaluate the credibility of sources or identify potential biases in information?  I can also provide general guidance on critical thinking and media literacy.  Please let me know how I can assist you further.  If you have any other questions or topics you'd like to discuss, feel free to ask!  I'm here to help.  Is there""")
false_text = ("""The claim \"Florida Democrats just voted to impose Sharia law on women\" is false. This conclusion can be drawn based on the evidence provided. Firstly, the Democrats voted against a bill that would have barred the use of foreign law in certain contexts, which is a far more limited measure than imposing Sharia law. Secondly, the bill in question only addressed contracts that select foreign law, and even then, a judge could still reject any foreign law that violates US public policy, such as child labor rules. Furthermore, the Democrats did vote against a bill that would have prohibited judges from applying foreign law in family-law cases if it contradicts US public policy, but this does not equate to imposing Sharia law. The evidence suggests that the Democrats were opposing bills that aimed to limit the use of foreign law, not imposing Sharia law on women. Therefore, the claim is unfounded and lacks merit.""")
gold_label, pred_label = "pants-fire", "pants-fire"

# 5) Load tokenizer & model
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large", num_labels=3, output_attentions=True
)
model = resize_position_embeddings(model, new_max_pos=1024)
checkpoint = torch.load(
    "best_model/best_llama_seed123_roberta.pth",
    map_location="cpu",
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# 6) Compute attention **separately** for each segment
sections = {}
sections["Claim"] = get_word_attentions(model, tokenizer, claim_text)
sections["Support Justifications"] = get_word_attentions(model, tokenizer, true_text)
sections["Refute Justifications"] = get_word_attentions(model, tokenizer, false_text)

# 7) Build HTML in the desired format
html = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Sample</title>
  <style>
    body { font-family: Arial; font-size:11px; margin:20px; }
    .section-title { margin:12px 0 4px; font-weight:bold; }
    .label { margin:4px 0; }
    .word { display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }
  </style>
</head>
<body>
"""

# Claim + labels
cw, cs = sections["Claim"]
thr = np.percentile(cs, 75)
mn, mx = min(cs), max(cs)
html += '<p class="section-title">Claim:</p>\n<p>“'
for w, sc in zip(cw, cs):
    norm = (sc - mn) / (mx - mn + 1e-12)
    if sc >= thr:
        color = get_color(norm)
        fg = "white" if norm > 0.5 else "black"
        html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
    else:
        html += f"{w} "
html += '”</p>\n'
html += f'<p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>\n'
html += f'<p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>\n'

# Justifications
for title in ["Support Justifications", "Refute Justifications"]:
    lw, ls = sections[title]
    thr = np.percentile(ls, 75)
    mn, mx = min(ls), max(ls)
    html += f'<p class="section-title">{title}:</p>\n<p>“'
    for w, sc in zip(lw, ls):
        norm = (sc - mn) / (mx - mn + 1e-12)
        if sc >= thr:
            color = get_color(norm)
            fg = "white" if norm > 0.5 else "black"
            html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
        else:
            html += f"{w} "
    html += '”</p>\n'

html += "</body></html>"

# 8) Save HTML + single‐page PDF
with open("attention_sample.html", "w", encoding="utf-8") as f:
    f.write(html)
HTML("attention_sample.html").write_pdf("attention_sample_22.pdf")

print("✅ attention_sample.html & attention_sample.pdf generated.")


In [ ]:
import torch
import re
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from weasyprint import HTML
import string
SEP = tokenizer.sep_token
# 1) Resize positional embeddings
def resize_position_embeddings(model, new_max_pos=1024):
    curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > curr_max:
        new_pos = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos.weight.data[curr_max:] = (
            model.roberta.embeddings.position_embeddings.weight.data[-1]
            .unsqueeze(0)
            .repeat(new_max_pos - curr_max, 1)
        )
        model.roberta.embeddings.position_embeddings = new_pos
        model.config.max_position_embeddings = new_max_pos
    return model

# 2) Word-level attention extraction
def get_word_attentions_with_spans(model, tokenizer, text):
    """
    Returns three lists:
     - words: the exact substrings from `text`
     - scores: the average [CLS]-attention for each substring
     - spans:  the (start, end) character indices in `text`
    Filters out any pure separators or punctuation spans.
    """
    SEP = tokenizer.sep_token  # e.g. "</s>"

    # Tokenize and get offsets
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    ids, mask = enc["input_ids"], enc["attention_mask"]

    # Forward pass to get attentions
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask)
        attn = out.attentions[-1]           # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]     # (L, L)
    cls_attn = mean_attn[:, 0].tolist()     # (L,)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])

    words, scores, spans = [], [], []
    last_end = -1
    grp_start = grp_end = None
    grp_scores = []

    for tok, (start, end), score in zip(tokens, offsets, cls_attn):
        # Skip special & pad tokens
        if tok in tokenizer.all_special_tokens:
            continue

        # If this token does not continue the previous span → flush old group
        if start != last_end:
            if grp_start is not None:
                substring = text[grp_start:grp_end]
                # filter out separator or angle-bracket artifacts
                if SEP in substring or '<' in substring or '>' in substring:
                    pass
                # only keep if contains at least one alphanumeric
                elif re.search(r'\w', substring):
                    words.append(substring)
                    scores.append(sum(grp_scores) / len(grp_scores))
                    spans.append((grp_start, grp_end))
            # start new group
            grp_start, grp_end = start, end
            grp_scores = [score]
        else:
            # continuation of same word
            grp_end = end
            grp_scores.append(score)

        last_end = end

    # flush final group
    if grp_start is not None:
        substring = text[grp_start:grp_end]
        if not (SEP in substring or '<' in substring or '>' in substring) and re.search(r'\w', substring):
            words.append(substring)
            scores.append(sum(grp_scores) / len(grp_scores))
            spans.append((grp_start, grp_end))

    return words, scores, spans

# 3) Color mapping
def get_color(score):
    pct = (1 - score) * 100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Texts & labels
claim_text = (
    """Police pepper-sprayed a double amputee and removed his prosthetic legs during a June 21, 2020, protest against police brutality in Columbus, Ohio. """
)
true_text = (
    """Based on the provided evidence, I conclude that the statement is TRUE. The evidence consists of multiple reports, videos, and witness accounts from various sources, including Twitter users, news outlets, and the Columbus mayor. These accounts consistently describe the incident where a double amputee was pepper-sprayed by police and had his prosthetic legs removed during a protest in Columbus, Ohio, on June 21, 2020. The details of the incident, including the removal of the prosthetic legs and the officer's identity, are corroborated across multiple sources, lending credibility to the reports.
    The widespread outrage and public attention to the incident further support the validity of the statement."""
)
false_text = (
    """The claim that police pepper-sprayed a double amputee and removed his prosthetic legs during a June 21, 2020, protest against police brutality in Columbus, Ohio, is false. This conclusion is supported by multiple lines of evidence. Firstly, the Columbus Police Department disputes the claim, stating that the individual attacked the officers and was carried away by protesters. This is corroborated by police body camera footage and a pole-mounted video camera at the scene, which show the individual throwing a sign and a bottle of liquid at the officers. Secondly, there is no footage of the prosthetic leg being removed from the man, who returns a short time later with his leg attached, according to police video. This suggests that the prosthetic leg was not taken by the police. Thirdly, witnesses from the scene describe the man crawling on his hands to get medical help, and a group of protesters rushing the officer to get his leg back. However, the police video shows that the protesters were the ones who pulled the man away from the officer, causing him to lose his prosthetic leg. Lastly, the police have evidence that shows the individual attacked the officers, which contradicts the claim that the police removed the prosthetic leg without provocation. In conclusion, the evidence suggests that the claim is false, and the police did not remove the prosthetic leg from the double amputee. The incident was a result of a violent clash between the police and protesters, and the man's prosthetic leg was lost due to the actions of the protesters, not the police."""
)

gold_label, pred_label = "half", "half"

# 5) Load model & tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large", num_labels=3, output_attentions=True
)
model = resize_position_embeddings(model, new_max_pos=1024)
ckpt = torch.load(
    "best_model/best_llama_seed123_roberta.pth",
    map_location="cpu",
    weights_only=False,
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 6) Concatenate & extract from combined
sep       = tokenizer.sep_token
combined  = f"{claim_text} {sep} {true_text} {sep} {false_text}"
words, scores, spans = get_word_attentions_with_spans(model, tokenizer, combined)

# find sep positions in the combined string
i1 = combined.find(sep)
i2 = combined.find(sep, i1 + len(sep))
end1 = i1 + len(sep)
end2 = i2 + len(sep)

# 7) Split into three lists by span
claim_words,  claim_scores  = [], []
true_words,   true_scores   = [], []
false_words,  false_scores  = [], []

for w, sc, (st, en) in zip(words, scores, spans):
    if en <= i1:
        claim_words.append(w);  claim_scores.append(sc)
    elif st >= end1 and en <= i2:
        true_words.append(w);   true_scores.append(sc)
    elif st >= end2:
        false_words.append(w);  false_scores.append(sc)
    # else: it’s part of the separator; skip

sections = {
    "Claim":               (claim_words,  claim_scores),
    "Support Justifications": (true_words,  true_scores),
    "Refute Justifications":(false_words, false_scores),
}


# 7) Build HTML (no borders, all sections highlighted)
html = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Sample</title>
  <style>
    body { font-family: Arial; font-size:11px; margin:20px; }
    .section-title { margin:12px 0 4px; font-weight:bold; }
    .label { margin:4px 0; }
    .word { display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }
  </style>
</head>
<body>
"""

# Add Claim & labels
html += f'<p class="section-title">Claim:</p>\n<p>“'
# highlight top-25% in claim
cw_words, cw_scores = sections["Claim"]
thr_c = np.percentile(cw_scores, 75)
mn_c, mx_c = min(cw_scores), max(cw_scores)
for w, sc in zip(cw_words, cw_scores):
    norm = (sc - mn_c) / (mx_c - mn_c + 1e-12)
    if sc >= thr_c:
        color = get_color(norm)
        fg    = "white" if norm>0.5 else "black"
        html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
    else:
        html += f"{w} "
html += '”</p>\n'

html += f'<p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>\n'
html += f'<p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>\n'

# Add True/False Justifications
for title in ["Support Justifications", "Refute Justifications"]:
    words, scores = sections[title]
    thr = np.percentile(scores, 75)
    mn, mx = min(scores), max(scores)

    html += f'<p class="section-title">{title}:</p>\n<p>“'
    for w, sc in zip(words, scores):
        norm = (sc - mn) / (mx - mn + 1e-12)
        if sc >= thr:
            color = get_color(norm)
            fg    = "white" if norm>0.5 else "black"
            html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
        else:
            html += f"{w} "
    html += '”</p>\n'

html += "</body></html>"

# 8) Save HTML + PDF
with open("attention_sample.html", "w", encoding="utf-8") as f:
    f.write(html)
HTML("attention_sample.html").write_pdf("attention_sample_15.1.pdf")

print("✅ attention_sample.html & attention_sample_15.pdf generated.")


## XLNET

In [ ]:
import torch
import numpy as np
from transformers import XLNetTokenizerFast, XLNetForSequenceClassification
from weasyprint import HTML

# 1) (Removed) RoBERTa positional‐embedding resizing – XLNet-large supports up to 1024 tokens by default

# 2) Compute word‐level attention scores
def get_word_attentions_with_spans(model, tokenizer, text):
    """
    Returns three lists:
     - words: the exact substrings from `text`
     - scores: the average [CLS]-attention for each substring
     - spans:  the (start, end) character indices in `text`
    Filters out any pure separators or punctuation spans.
    """
    SEP = tokenizer.sep_token  # e.g. "</s>"

    # Tokenize and get offsets
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    ids, mask = enc["input_ids"], enc["attention_mask"]

    # Forward pass to get attentions
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask)
        attn = out.attentions[-1]           # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]     # (L, L)
    cls_attn = mean_attn[:, 0].tolist()     # (L,)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])

    words, scores, spans = [], [], []
    last_end = -1
    grp_start = grp_end = None
    grp_scores = []

    for tok, (start, end), score in zip(tokens, offsets, cls_attn):
        # Skip special & pad tokens
        if tok in tokenizer.all_special_tokens:
            continue

        # If this token does not continue the previous span → flush old group
        if start != last_end:
            if grp_start is not None:
                substring = text[grp_start:grp_end]
                # filter out separator or angle-bracket artifacts
                if SEP in substring or '<' in substring or '>' in substring:
                    pass
                # only keep if contains at least one alphanumeric
                elif re.search(r'\w', substring):
                    words.append(substring)
                    scores.append(sum(grp_scores) / len(grp_scores))
                    spans.append((grp_start, grp_end))
            # start new group
            grp_start, grp_end = start, end
            grp_scores = [score]
        else:
            # continuation of same word
            grp_end = end
            grp_scores.append(score)

        last_end = end

    # flush final group
    if grp_start is not None:
        substring = text[grp_start:grp_end]
        if not (SEP in substring or '<' in substring or '>' in substring) and re.search(r'\w', substring):
            words.append(substring)
            scores.append(sum(grp_scores) / len(grp_scores))
            spans.append((grp_start, grp_end))

    return words, scores, spans

# 3) Color mapping
def get_color(score):
    pct = (1 - score) * 100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Define your segments and labels
claim_text = (
    "Foreign aid is less than 1 percent of our federal budget."
)

true_text = (
    """The statement that foreign aid is less than 1 percent of our federal budget is supported
by multiple sources. According to various statements, foreign aid makes up less than one
percent of the total federal budget, is a \"drop in the budgetary bucket,\" and amounts to
far less than what most Americans think is spent on it. Additionally, it’s stated that less
than 1 percent of the $ 4 trillion federal budget goes to foreign aid, and Americans actually
spend less than 1 percent of the budget on foreign aid annually. These consistent claims from
different sources suggest that foreign aid is indeed a relatively small portion of the federal
budget. The evidence collectively supports this assertion. <|im_end|>"""
)
false_text = (
    """he claim that foreign aid is less than 1 percent of our federal budget appears to be misleading.
While it is true that foreign aid accounts for a relatively small portion of the federal budget,
the actual figure is closer to 1% of the discretionary budget, which is a subset of the overall
federal budget. Moreover, the claim ignores the fact that foreign aid has been steadily
decreasing as a percentage of the federal budget over the years. For instance, in 2019, foreign
assistance accounted for approximately 0.9% of the federal budget, but this figure has been
declining since the 1960s. Furthermore, the claim fails to consider the impact of foreign
aid on global development and poverty reduction, which is a critical aspect of U.S. foreign
policy. In reality, foreign aid plays a vital role in addressing pressing global issues such as
poverty, hunger, and disease, and its benefits extend far beyond the 1% of the federal budget
allocated to it. Therefore, the claim that foreign aid is less than 1 percent of our federal
budget is an oversimplification that does not accurately reflect the complexity of the issue."""
)

gold_label, pred_label = "true", "true"

# 5) Load tokenizer & model
tokenizer = XLNetTokenizerFast.from_pretrained("xlnet-large-cased")
model = XLNetForSequenceClassification.from_pretrained(
    "xlnet-large-cased",
    num_labels=6,
    output_attentions=True
)
# no resizing step for XLNet
checkpoint = torch.load(
    "best_model/best_llama_seed42_xlnet.pth",  # point to your XLNet checkpoint
    map_location="cpu",
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

sep       = tokenizer.sep_token
combined  = f"{claim_text} {sep} {true_text} {sep} {false_text}"
words, scores, spans = get_word_attentions_with_spans(model, tokenizer, combined)

# find sep positions in the combined string
i1 = combined.find(sep)
i2 = combined.find(sep, i1 + len(sep))
end1 = i1 + len(sep)
end2 = i2 + len(sep)

# 7) Split into three lists by span
claim_words,  claim_scores  = [], []
true_words,   true_scores   = [], []
false_words,  false_scores  = [], []

for w, sc, (st, en) in zip(words, scores, spans):
    if en <= i1:
        claim_words.append(w);  claim_scores.append(sc)
    elif st >= end1 and en <= i2:
        true_words.append(w);   true_scores.append(sc)
    elif st >= end2:
        false_words.append(w);  false_scores.append(sc)
    # else: it’s part of the separator; skip

sections = {
    "Claim":               (claim_words,  claim_scores),
    "Support Justifications": (true_words,  true_scores),
    "Refute Justifications":(false_words, false_scores),
}


# 7) Build HTML (no borders, all sections highlighted)
html = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Sample</title>
  <style>
    body { font-family: Arial; font-size:11px; margin:20px; }
    .section-title { margin:12px 0 4px; font-weight:bold; }
    .label { margin:4px 0; }
    .word { display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }
  </style>
</head>
<body>
"""

# Add Claim & labels
html += f'<p class="section-title">Claim:</p>\n<p>“'
# highlight top-25% in claim
cw_words, cw_scores = sections["Claim"]
thr_c = np.percentile(cw_scores, 75)
mn_c, mx_c = min(cw_scores), max(cw_scores)
for w, sc in zip(cw_words, cw_scores):
    norm = (sc - mn_c) / (mx_c - mn_c + 1e-12)
    if sc >= thr_c:
        color = get_color(norm)
        fg    = "white" if norm>0.5 else "black"
        html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
    else:
        html += f"{w} "
html += '”</p>\n'

html += f'<p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>\n'
html += f'<p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>\n'

# Add True/False Justifications
for title in ["Support Justifications", "Refute Justifications"]:
    words, scores = sections[title]
    thr = np.percentile(scores, 75)
    mn, mx = min(scores), max(scores)

    html += f'<p class="section-title">{title}:</p>\n<p>“'
    for w, sc in zip(words, scores):
        norm = (sc - mn) / (mx - mn + 1e-12)
        if sc >= thr:
            color = get_color(norm)
            fg    = "white" if norm>0.5 else "black"
            html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
        else:
            html += f"{w} "
    html += '”</p>\n'

html += "</body></html>"

# 8) Save HTML + PDF
with open("attention_sample.html", "w", encoding="utf-8") as f:
    f.write(html)
HTML("attention_sample.html").write_pdf("attention_sample_tabel_17.pdf")

print("✅ attention_sample.html & attention_sample.pdf generated.")


## lair_raw_examples

In [ ]:
import torch
import re
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from weasyprint import HTML
import string
SEP = tokenizer.sep_token
# 1) Resize positional embeddings
def resize_position_embeddings(model, new_max_pos=1024):
    curr_max, embed_size = model.roberta.embeddings.position_embeddings.weight.shape
    if new_max_pos > curr_max:
        new_pos = torch.nn.Embedding(new_max_pos, embed_size)
        new_pos.weight.data[:curr_max] = model.roberta.embeddings.position_embeddings.weight.data
        new_pos.weight.data[curr_max:] = (
            model.roberta.embeddings.position_embeddings.weight.data[-1]
            .unsqueeze(0)
            .repeat(new_max_pos - curr_max, 1)
        )
        model.roberta.embeddings.position_embeddings = new_pos
        model.config.max_position_embeddings = new_max_pos
    return model

# 2) Word-level attention extraction
def get_word_attentions_with_spans(model, tokenizer, text):
    """
    Returns three lists:
     - words: the exact substrings from `text`
     - scores: the average [CLS]-attention for each substring
     - spans:  the (start, end) character indices in `text`
    Filters out any pure separators or punctuation spans.
    """
    SEP = tokenizer.sep_token  # e.g. "</s>"

    # Tokenize and get offsets
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    ids, mask = enc["input_ids"], enc["attention_mask"]

    # Forward pass to get attentions
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask)
        attn = out.attentions[-1]           # (1, heads, L, L)
        mean_attn = attn.mean(dim=1)[0]     # (L, L)
    cls_attn = mean_attn[:, 0].tolist()     # (L,)

    tokens = tokenizer.convert_ids_to_tokens(ids[0])

    words, scores, spans = [], [], []
    last_end = -1
    grp_start = grp_end = None
    grp_scores = []

    for tok, (start, end), score in zip(tokens, offsets, cls_attn):
        # Skip special & pad tokens
        if tok in tokenizer.all_special_tokens:
            continue

        # If this token does not continue the previous span → flush old group
        if start != last_end:
            if grp_start is not None:
                substring = text[grp_start:grp_end]
                # filter out separator or angle-bracket artifacts
                if SEP in substring or '<' in substring or '>' in substring:
                    pass
                # only keep if contains at least one alphanumeric
                elif re.search(r'\w', substring):
                    words.append(substring)
                    scores.append(sum(grp_scores) / len(grp_scores))
                    spans.append((grp_start, grp_end))
            # start new group
            grp_start, grp_end = start, end
            grp_scores = [score]
        else:
            # continuation of same word
            grp_end = end
            grp_scores.append(score)

        last_end = end

    # flush final group
    if grp_start is not None:
        substring = text[grp_start:grp_end]
        if not (SEP in substring or '<' in substring or '>' in substring) and re.search(r'\w', substring):
            words.append(substring)
            scores.append(sum(grp_scores) / len(grp_scores))
            spans.append((grp_start, grp_end))

    return words, scores, spans

# 3) Color mapping
def get_color(score):
    pct = (1 - score) * 100
    return f"hwb(200 {pct:.1f}% 20%)"

# 4) Texts & labels
claim_text = (
    "Florida Democrats just voted to impose Sharia law on women."
)

true_text = ("""I can't help you with this request. I can’t verify the accuracy of claims about imposing Sharia law on women in Florida. Is there anything else I can help you with?  Would you like to know how to evaluate the credibility of sources or identify potential biases in information?  I can also provide general guidance on critical thinking and media literacy.  Please let me know how I can assist you further.  If you have any other questions or topics you'd like to discuss, feel free to ask!  I'm here to help.  Is there anything else I can help you with?  Would you like to know more about fact-checking or evaluating evidence?  I can also provide information on how to identify reliable sources and avoid misinformation.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about critical thinking or how to evaluate information?  I can also provide information on how to identify potential biases and propaganda.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about how to evaluate the credibility of sources or identify potential biases in information?  I can also provide general guidance on critical thinking and media literacy.  Please let me know how I can assist you further.  If you have any other questions or topics you'd like to discuss, feel free to ask!  I'm here to help.  Is there anything else I can help you with?  Would you like to know more about fact-checking or evaluating evidence?  I can also provide information on how to identify reliable sources and avoid misinformation.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about critical thinking or how to evaluate information?  I can also provide information on how to identify potential biases and propaganda.  Let me know if there's anything else I can help you with.  Is there anything else I can help you with?  Would you like to know more about how to evaluate the credibility of sources or identify potential biases in information?  I can also provide general guidance on critical thinking and media literacy.  Please let me know how I can assist you further.  If you have any other questions or topics you'd like to discuss, feel free to ask!  I'm here to help.  Is there""")
false_text = ("""The claim \"Florida Democrats just voted to impose Sharia law on women\" is false. This conclusion can be drawn based on the evidence provided. Firstly, the Democrats voted against a bill that would have barred the use of foreign law in certain contexts, which is a far more limited measure than imposing Sharia law. Secondly, the bill in question only addressed contracts that select foreign law, and even then, a judge could still reject any foreign law that violates US public policy, such as child labor rules. Furthermore, the Democrats did vote against a bill that would have prohibited judges from applying foreign law in family-law cases if it contradicts US public policy, but this does not equate to imposing Sharia law. The evidence suggests that the Democrats were opposing bills that aimed to limit the use of foreign law, not imposing Sharia law on women. Therefore, the claim is unfounded and lacks merit.""")
gold_label, pred_label = "pants-fire", "pants-fire"

# 5) Load model & tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large", num_labels=3, output_attentions=True
)
model = resize_position_embeddings(model, new_max_pos=1024)
ckpt = torch.load(
    "best_model/best_llama_seed123_roberta.pth",
    map_location="cpu",
    weights_only=False,
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 6) Concatenate & extract from combined
sep       = tokenizer.sep_token
combined  = f"{claim_text} {sep} {true_text} {sep} {false_text}"
words, scores, spans = get_word_attentions_with_spans(model, tokenizer, combined)

# find sep positions in the combined string
i1 = combined.find(sep)
i2 = combined.find(sep, i1 + len(sep))
end1 = i1 + len(sep)
end2 = i2 + len(sep)

# 7) Split into three lists by span
claim_words,  claim_scores  = [], []
true_words,   true_scores   = [], []
false_words,  false_scores  = [], []

for w, sc, (st, en) in zip(words, scores, spans):
    if en <= i1:
        claim_words.append(w);  claim_scores.append(sc)
    elif st >= end1 and en <= i2:
        true_words.append(w);   true_scores.append(sc)
    elif st >= end2:
        false_words.append(w);  false_scores.append(sc)
    # else: it’s part of the separator; skip

sections = {
    "Claim":               (claim_words,  claim_scores),
    "Support Justifications": (true_words,  true_scores),
    "Refute Justifications":(false_words, false_scores),
}


# 7) Build HTML (no borders, all sections highlighted)
html = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Attention Sample</title>
  <style>
    body { font-family: Arial; font-size:11px; margin:20px; }
    .section-title { margin:12px 0 4px; font-weight:bold; }
    .label { margin:4px 0; }
    .word { display:inline-block; padding:2px 4px; margin:1px; border-radius:3px; word-break:break-word; }
  </style>
</head>
<body>
"""

# Add Claim & labels
html += f'<p class="section-title">Claim:</p>\n<p>“'
# highlight top-25% in claim
cw_words, cw_scores = sections["Claim"]
thr_c = np.percentile(cw_scores, 75)
mn_c, mx_c = min(cw_scores), max(cw_scores)
for w, sc in zip(cw_words, cw_scores):
    norm = (sc - mn_c) / (mx_c - mn_c + 1e-12)
    if sc >= thr_c:
        color = get_color(norm)
        fg    = "white" if norm>0.5 else "black"
        html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
    else:
        html += f"{w} "
html += '”</p>\n'

html += f'<p class="label"><strong>Gold Label:</strong> “{gold_label}”</p>\n'
html += f'<p class="label"><strong>Predicted Label:</strong> “{pred_label}”</p>\n'

# Add True/False Justifications
for title in ["Support Justifications", "Refute Justifications"]:
    words, scores = sections[title]
    thr = np.percentile(scores, 75)
    mn, mx = min(scores), max(scores)

    html += f'<p class="section-title">{title}:</p>\n<p>“'
    for w, sc in zip(words, scores):
        norm = (sc - mn) / (mx - mn + 1e-12)
        if sc >= thr:
            color = get_color(norm)
            fg    = "white" if norm>0.5 else "black"
            html += f'<span class="word" style="background:{color};color:{fg}">{w}</span> '
        else:
            html += f"{w} "
    html += '”</p>\n'

html += "</body></html>"

# 8) Save HTML + PDF
with open("attention_sample.html", "w", encoding="utf-8") as f:
    f.write(html)
HTML("attention_sample.html").write_pdf("attention_sample_tabel_22.pdf")

print(" attention_sample.html & attention_sample_15.pdf generated.")
